# Llama 3.2 1B chat in your Google Colab

Runs on Google's free GPU in **your** Google account, so nothing is downloaded to your phone or laptop.

1. **Runtime > Change runtime type > T4 GPU** (faster; CPU works too).
2. Run the cells top to bottom (the play button on each).
3. Chat in step 3.

Uses `unsloth/Llama-3.2-1B-Instruct`, an ungated copy of Meta's model, so you don't need a Hugging Face login. Llama 3.2 Community License applies.

In [ ]:
#@title 1. Keep the model in your Google Drive (optional)
#@markdown Saves the download in `MyDrive/ML-HOST/models`, so next time it loads without downloading again. Untick to skip Drive.
use_drive = True #@param {type:"boolean"}
import os
cache = "/content/hf"
if use_drive:
    from google.colab import drive
    drive.mount("/content/drive")
    cache = "/content/drive/MyDrive/ML-HOST/models"
os.makedirs(cache, exist_ok=True)
os.environ["HF_HOME"] = cache  # must be set before transformers is imported


In [ ]:
#@title 2. Load the model
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
MODEL = "unsloth/Llama-3.2-1B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL).to(device)
if device == "cuda":
    model = model.half()
print("Loaded on", device.upper(), "" if device == "cuda" else "(slow: Runtime > Change runtime type > T4 GPU)")


In [ ]:
#@title 3. Chat
#@markdown Type a message and press Enter. Send an empty message to stop; run this cell again to start a new chat.
history = []
while (q := input("You: ").strip()):
    history.append({"role": "user", "content": q})
    text = tok.apply_chat_template(history, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(text, return_tensors="pt").to(device)
    out = model.generate(**inputs, max_new_tokens=512, do_sample=True, temperature=0.7)
    answer = tok.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    history.append({"role": "assistant", "content": answer})
    print("Model:", answer, "\n")
